In [8]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

In [9]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [10]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [11]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [12]:
prompt = "She completed the project in under 40 days"

with model.trace(prompt):
    activations = model.transformer.h[layer].input[0][0]

    middle = sae_list[0](activations)

    acts = middle[1]
    acts.save()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [13]:
from circuitsvis.tokens import colored_tokens

vis_acts = acts[0,:,4]
vis_acts[0] = 0
toks = model.tokenizer.encode(prompt)
str_toks = [model.tokenizer.decode([tok]) for tok in toks]

colored_tokens(str_toks, vis_acts)

In [14]:
toks

[3347, 5668, 262, 1628, 287, 739, 2319, 1528]